# Learning Is Optimization: Linear and Logistic Regression, Trained by Gradient Descent

The runnable code of [Learning Is Optimization: Linear and Logistic Regression, Trained by Gradient Descent](https://residual-stream.lvtan71.workers.dev/posts/learning-is-optimization/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/learning-is-optimization.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "learning-is-optimization"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/learning-is-optimization/a_loss.py', 'shared/learning-is-optimization/c_router.py']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Choosing what "wrong" means: L2 and L1 loss

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/learning-is-optimization/#choosing-what-wrong-means-l2-and-l1-loss)

In [ ]:
%%writefile a_loss.py
import numpy as np

# Words and o200k_base tokens for each English post on this blog, 2026-09-29
# (full MDX body, including math and code).
posts = {
    "attention-alternatives": (5880, 9793),
    "bpe-tokenizer-explained": (6638, 11489),
    "context-engineering-compaction-clearing-memory": (4336, 7104),
    "disciplined-ml-chess-outcomes": (2634, 4486),
    "kv-cache-explained": (3227, 5737),
    "paged-attention-explained": (2716, 4464),
    "residual-stream-explained": (2815, 4942),
    "sglang-radixattention-explained": (3143, 5140),
    "tensorrt-llm-explained": (3435, 6199),
    "transformer-attention-patterns": (3507, 6808),
    "vllm-architecture-explained": (2584, 4405),
    "vllm-vs-sglang-vs-tensorrt-llm": (1732, 3209),
}
words = np.array([w for w, _ in posts.values()]) / 1000   # thousands
tokens = np.array([t for _, t in posts.values()]) / 1000

# The simplest possible model: every post costs c tokens per word.
ratio = tokens / words

if __name__ == "__main__":
    mse = lambda c: ((ratio - c) ** 2).mean()
    mae = lambda c: np.abs(ratio - c).mean()
    grid = np.linspace(1.5, 2.0, 50001)
    print(f"L2 minimizer = {grid[np.argmin([mse(c) for c in grid])]:.4f}"
          f"   mean   = {ratio.mean():.4f}")
    lo, hi = np.sort(ratio)[5], np.sort(ratio)[6]   # the two middle values
    print(f"L1 is flat on [{lo:.4f}, {hi:.4f}]:"
          f" MAE = {mae(lo):.5f}, {mae((lo + hi) / 2):.5f}, {mae(hi):.5f}")
    print(f"median = {np.median(ratio):.4f}")

In [ ]:
%run a_loss.py

The post printed:

```text
L2 minimizer = 1.7378   mean   = 1.7378
L1 is flat on [1.7047, 1.7308]: MAE = 0.07269, 0.07269, 0.07269
median = 1.7178
```

## How big a step is safe?

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/learning-is-optimization/#how-big-a-step-is-safe)

In [ ]:
import numpy as np
from a_loss import words as x, tokens as y

def gradient_descent(x, y, eta, steps):
    w, b = 0.0, 0.0
    for t in range(steps):
        err = w * x + b - y               # prediction minus target
        w -= eta * 2 * (err * x).mean()   # dL/dw
        b -= eta * 2 * err.mean()         # dL/db
    return w, b, ((w * x + b - y) ** 2).mean()

for eta in [0.01, 0.05, 0.07]:
    w, b, mse = gradient_descent(x, y, eta, steps=400)
    print(f"raw words, eta={eta:<5}: w={w:9.4g}  b={b:9.4g}  MSE={mse:.4g}")

xs = (x - x.mean()) / x.std()             # standardize the feature
for eta in [0.5, 1.0, 1.05]:
    w, b, mse = gradient_descent(xs, y, eta, steps=400)
    print(f"standardized, eta={eta:<4}: w={w:9.4g}  b={b:9.4g}  MSE={mse:.4g}")

The post printed:

```text
raw words, eta=0.01 : w=    1.654  b=   0.2806  MSE=0.08979
raw words, eta=0.05 : w=    1.672  b=   0.2073  MSE=0.08908
raw words, eta=0.07 : w=-3.273e+24  b=-8.102e+23  MSE=1.746e+50
standardized, eta=0.5 : w=    2.273  b=    6.148  MSE=0.08908
standardized, eta=1.0 : w=-6.963e-13  b=-3.979e-13  MSE=43.05
standardized, eta=1.05: w=-8.198e+16  b=-2.217e+17  MSE=5.588e+34
```

## Training the router

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/learning-is-optimization/#training-the-router)

In [ ]:
%%writefile c_router.py
import re
import numpy as np

SERVING = {"kv", "cache", "caching", "batch", "batching", "latency",
           "throughput", "serve", "serving", "vllm", "sglang", "tensorrt",
           "gpu", "decode", "prefill", "paged", "quantization", "memory"}

questions = [  # (question, 1 = send to the serving posts)
    ("How big is the KV cache for a 70B model?", 1),
    ("Why is decode slower per token than prefill?", 1),
    ("Does vLLM use paged attention?", 1),
    ("How does continuous batching raise throughput?", 1),
    ("What limits the batch size on one GPU?", 1),
    ("Is SGLang faster than TensorRT-LLM?", 1),
    ("Why is my chatbot slow to answer long prompts?", 1),
    ("How much memory does speculative decoding save?", 1),
    ("What does quantization do to latency?", 1),
    ("Why does the first token take so long?", 1),
    ("How do I serve a model to many users?", 1),
    ("Where does prefix caching help?", 1),
    ("How does backpropagation work?", 0),
    ("What is a residual stream?", 0),
    ("Why scale attention by the square root of d_k?", 0),
    ("How does BPE pick which pair to merge?", 0),
    ("What is the difference between an agent and a workflow?", 0),
    ("How does the memory tool in context engineering work?", 0),
    ("How much GPU memory does training a 7B model need?", 0),
    ("What does compaction keep from the transcript?", 0),
    ("Why did the chess model need a leakage contract?", 0),
    ("How is softmax computed?", 0),
    ("What does the tokenizer do with emoji?", 0),
    ("Should I cache the embeddings of every post?", 0),
]

def hits(q):
    return sum(tok in SERVING for tok in re.findall(r"[a-z0-9]+", q.lower()))

x = np.array([hits(q) for q, _ in questions], dtype=float)
y = np.array([label for _, label in questions], dtype=float)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

if __name__ == "__main__":
    w, b, eta = 0.0, 0.0, 0.5
    for step in range(5000):
        p = sigmoid(w * x + b)
        w -= eta * ((p - y) * x).mean()      # the whole gradient: (p - y) x
        b -= eta * (p - y).mean()

    p = sigmoid(w * x + b)
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
    print(f"w = {w:.3f}, b = {b:.3f}, log-loss = {loss:.4f}")
    for k in range(3):
        print(f"P(serving | {k} keyword hits) = {sigmoid(w * k + b):.3f}")
    print(f"accuracy = {((p >= 0.5) == y).mean():.3f}")

In [ ]:
%run c_router.py

The post printed:

```text
w = 1.746, b = -1.460, log-loss = 0.4837
P(serving | 0 keyword hits) = 0.188
P(serving | 1 keyword hits) = 0.571
P(serving | 2 keyword hits) = 0.884
accuracy = 0.792
```

In [ ]:
import numpy as np
from c_router import x, y, sigmoid

# Start confidently wrong (w = -5) and train with each loss, same steps.
for name in ["log-loss", "squared error"]:
    w, b = -5.0, 0.0
    for step in range(500):
        p = sigmoid(w * x + b)
        grad_z = (p - y) if name == "log-loss" else 2 * (p - y) * p * (1 - p)
        w -= 0.5 * (grad_z * x).mean()
        b -= 0.5 * grad_z.mean()
    p = sigmoid(w * x + b)
    print(f"{name:13s} after 500 steps: w = {w:6.3f}, b = {b:6.3f},"
          f" accuracy = {((p >= 0.5) == y).mean():.3f}")

The post printed:

```text
log-loss      after 500 steps: w =  1.746, b = -1.460, accuracy = 0.792
squared error after 500 steps: w = -4.869, b = -1.480, accuracy = 0.500
```

## One example at a time: stochastic and mini-batch descent

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/learning-is-optimization/#one-example-at-a-time-stochastic-and-mini-batch-descent)

In [ ]:
import numpy as np
from c_router import x, y, sigmoid

def log_loss(w, b):
    p = sigmoid(w * x + b)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

best = log_loss(1.746, -1.460)          # the full-batch optimum from above
for batch in [24, 4, 1]:
    finals, needed = [], []
    for seed in range(200):             # 200 different shuffles
        rng = np.random.default_rng(seed)
        w = b = 0.0
        updates, reached = 0, None
        while updates < 5000:
            order = rng.permutation(len(y))
            for start in range(0, len(y), batch):
                i = order[start:start + batch]
                p = sigmoid(w * x[i] + b)
                w -= 0.5 * ((p - y[i]) * x[i]).mean()
                b -= 0.5 * (p - y[i]).mean()
                updates += 1
                if reached is None and log_loss(w, b) < 1.01 * best:
                    reached = updates
        finals.append(log_loss(w, b)); needed.append(reached)
    finals = np.array(finals)
    print(f"{batch:2d} per update: updates to reach 1% of best, median {np.median(needed):.0f};"
          f"  final loss median {np.median(finals):.4f}, worst {finals.max():.4f}")

The post printed:

```text
24 per update: updates to reach 1% of best, median 52;  final loss median 0.4837, worst 0.4837
 4 per update: updates to reach 1% of best, median 52;  final loss median 0.4841, worst 0.4921
 1 per update: updates to reach 1% of best, median 44;  final loss median 0.4949, worst 0.7688
```